# 🚛 Fleet Management — Nostradamus IoTO SDK

Vehicle fleet telemetry: trucks, vans and buses report GPS position, speed, heading, fuel level and consumption, engine RPM and temperature, odometer, tyre pressure, harsh-braking counts and cargo weight.

This notebook mirrors `examples/smart_transportation/fleet_management.py` using only `nostradamus_ioto_sdk.NostradamusClient`.
It sends **5 vehicles × 24 hourly readings** (120 records) to the collection
`fleet_telemetry`, then queries, aggregates and deletes part of it.

## Workflow

1. Create the `fleet_telemetry` collection with a schema
2. Generate and send 5 vehicles × 24 hourly readings
3. Query speeding, overheating, low fuel and fuel-consumption ranking
4. Aggregate speed, engine temperature and fuel-level statistics
5. Delete a 12-hour window for `TRUCK_001`
6. Delete the collection

## What this notebook highlights

- Vehicle-type-specific speed and cargo profiles
- Speeding (> 90 km/h) and overheating (> 100 °C) detection
- Driver-behaviour signals (harsh braking, idle time)

## Prerequisites

```bash
pip install -e ../..          # from the repository root
pip install python-dotenv
```

| Variable | Purpose |
|----------|---------|
| `NOSTRADAMUS_PROJECT_ID` | Target project UUID |
| `NOSTRADAMUS_MASTER_KEY` | Master key (create/delete collection, delete data) |
| `NOSTRADAMUS_WRITE_KEY` | Write key (send data) |
| `NOSTRADAMUS_READ_KEY` | Read key (query data and statistics) |
| `NOSTRADAMUS_BASE_URL` | Optional API base URL override |

> The notebook is idempotent: if the collection already exists the SDK raises
> `APIError` with status `409`, and Step 1 falls back to reusing it.

In [ ]:
%pip install -q -e ../.. python-dotenv

## Configuration

Each permission level gets its own `NostradamusClient` instance — the SDK injects
the `X-API-Key` header, so there is no manual header handling.

In [ ]:
import os
import random
from collections import Counter
from datetime import datetime, timedelta, timezone

from dotenv import load_dotenv

from nostradamus_ioto_sdk import NostradamusClient, StatOperation
from nostradamus_ioto_sdk.exceptions import APIError

load_dotenv()

BASE_URL = os.getenv("NOSTRADAMUS_BASE_URL", "https://nostradamus-ioto.issel.ee.auth.gr")
PROJECT_ID = os.getenv("NOSTRADAMUS_PROJECT_ID", "")
MASTER_KEY = os.getenv("NOSTRADAMUS_MASTER_KEY", "")
WRITE_KEY = os.getenv("NOSTRADAMUS_WRITE_KEY", "")
READ_KEY = os.getenv("NOSTRADAMUS_READ_KEY", "")

_missing = [
    name
    for name, value in {
    "PROJECT_ID": PROJECT_ID,
    "MASTER_KEY": MASTER_KEY,
    "WRITE_KEY": WRITE_KEY,
    "READ_KEY": READ_KEY,
    }.items()
    if not value
]
if _missing:
    raise RuntimeError(
        "Missing required environment variables: " + ", ".join(_missing)
    )

print("🚛 Nostradamus IoTO SDK — Fleet Management")
print(f"base_url={BASE_URL} project_id={PROJECT_ID}")

## Step 1: Create the vehicle collection

`collections.create()` returns a `CollectionResponse`, so the collection ID comes
straight from the model. The schema declares one sample record and is copied
verbatim from `examples/smart_transportation/fleet_management.py`.

In [ ]:
COLLECTION_NAME = "fleet_telemetry"

COLLECTION_SCHEMA = {
                    "key": "TRUCK_001",
                    "timestamp": "2025-06-17T10:30:00Z",
                    "latitude": 40.60,
                    "longitude": 22.93,
                    "speed_kmh": 65,
                    "heading_deg": 180,
                    "fuel_level_pct": 72.0,
                    "fuel_consumption_lph": 12.5,
                    "engine_rpm": 2200,
                    "engine_temperature_c": 92.0,
                    "odometer_km": 85000.0,
                    "tire_pressure_bar": 3.2,
                    "harsh_braking_count": 0,
                    "idle_time_min": 5,
                    "cargo_weight_kg": 12000,
                    "battery_voltage_v": 13.8,
                }

with NostradamusClient(api_key=MASTER_KEY, base_url=BASE_URL) as master_client:
    try:
        collection = master_client.collections.create(
            project_id=PROJECT_ID,
            name=COLLECTION_NAME,
            description="Vehicle fleet GPS and diagnostics",
            tags=["fleet", "transportation", "telemetry"],
            collection_schema=COLLECTION_SCHEMA,
        )
        print("✅ Collection created")
    except APIError as exc:
        if exc.status_code != 409:
            raise
        collection = next(
            (
                coll
                for coll in master_client.collections.list(PROJECT_ID)
                if coll.collection_name == COLLECTION_NAME
            ),
            None,
        )
        if collection is None:
            raise
        print("⚠️  Collection already exists — reusing it")

collection_id = str(collection.collection_id)
print(f"collection_id={collection_id}")
collection.model_dump()

## Step 2: Generate and send fleet telemetry data

`data.send()` accepts a single dict or a list of dicts — the SDK serialises and
authenticates the batch with the write key. `generate_fleet_data()` is the
same generator used by the script.

In [ ]:
VEHICLE_PROFILES = {
    "TRUCK_001": {"type": "truck", "max_speed": 95, "max_cargo": 20000},
    "VAN_002": {"type": "van", "max_speed": 120, "max_cargo": 3000},
    "BUS_003": {"type": "bus", "max_speed": 100, "max_cargo": 0},
    "TRUCK_004": {"type": "truck", "max_speed": 90, "max_cargo": 18000},
    "VAN_005": {"type": "van", "max_speed": 115, "max_cargo": 2500},
}

def generate_fleet_data(num_readings=24):
    """Generate vehicle fleet telemetry data.

    Args:
        num_readings: Hourly readings per vehicle over 24h

    Returns:
        List of sensor readings
    """
    data = []
    base_time = datetime.now() - timedelta(hours=24)

    for vehicle_id, profile in VEHICLE_PROFILES.items():
        v_type = profile["type"]
        max_speed = profile["max_speed"]
        max_cargo = profile["max_cargo"]
        fuel_level = 85.0 + random.uniform(-15, 15)
        odometer = random.uniform(50000, 200000)
        base_lat = 40.60 + random.uniform(-0.02, 0.02)
        base_lon = 22.93 + random.uniform(-0.02, 0.02)

        for hour in range(num_readings):
            timestamp = base_time + timedelta(hours=hour)
            h = timestamp.hour

            is_driving = 6 <= h <= 22
            if not is_driving:
                speed = 0
            elif v_type == "truck":
                speed = random.randint(30, max_speed)
            elif v_type == "van":
                speed = random.randint(20, max_speed)
            else:
                speed = random.randint(15, max_speed)

            rpm = 600 if speed == 0 else 800 + speed * 30
            rpm = min(4500, rpm + random.randint(-200, 200))

            fuel_rate = 3.0 if speed == 0 else speed * 0.15
            fuel_rate += random.uniform(-1, 2)
            fuel_rate = round(max(3, min(25, fuel_rate)), 1)

            fuel_level -= fuel_rate * 0.05
            fuel_level = max(10, fuel_level)

            engine_temp = 80 + speed * 0.2 + random.uniform(-5, 10)
            engine_temp = round(min(110, max(75, engine_temp)), 1)

            odometer += speed * 0.8
            lat = base_lat + random.uniform(-0.03, 0.03)
            lon = base_lon + random.uniform(-0.03, 0.03)

            cargo = 0
            if v_type == "truck":
                cargo = random.randint(2000, max_cargo)
            elif v_type == "van":
                cargo = random.randint(200, max_cargo)

            harsh = (
                random.choices([0, 1, 2, 3], weights=[0.8, 0.1, 0.05, 0.05])[0]
                if is_driving
                else 0
            )

            idle_time = random.randint(0, 30) if is_driving else 0

            reading = {
                "key": vehicle_id,
                "timestamp": timestamp.strftime("%Y-%m-%dT%H:%M:%S"),
                "latitude": round(lat, 6),
                "longitude": round(lon, 6),
                "speed_kmh": speed,
                "heading_deg": random.randint(0, 359),
                "fuel_level_pct": round(fuel_level, 1),
                "fuel_consumption_lph": fuel_rate,
                "engine_rpm": rpm,
                "engine_temperature_c": engine_temp,
                "odometer_km": round(odometer, 1),
                "tire_pressure_bar": round(random.uniform(2.0, 3.8), 1),
                "harsh_braking_count": harsh,
                "idle_time_min": idle_time,
                "cargo_weight_kg": cargo,
                "battery_voltage_v": round(random.uniform(11.5, 14.5), 1),
            }
            data.append(reading)

    return data

data = generate_fleet_data()
print(f"Generated {len(data)} records")

with NostradamusClient(api_key=WRITE_KEY, base_url=BASE_URL) as write_client:
    write_client.data.send(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        data=data,
    )
print(f"✅ Sent {len(data)} records to {COLLECTION_NAME}")

## Step 3: Query data with domain filters

`data.get()` returns a `DataListResponse` with `.data` (records) and
`.total_count`. Filters are plain dicts combined as an AND. The queries below
mirror the alert conditions from the script:

- **🚚 TRUCK_001 readings**
- **💨 Speeding (> 90 km/h)**
- **🔥 Overheating (engine temp > 100 °C)**
- **⛽ Low fuel (< 20 %)**
- **Top 5 fuel consumption**

In [ ]:
def summary(result, label):
    """Print a compact view of a DataListResponse."""
    print(f"{label}: {len(result.data)} of {result.total_count} record(s)")
    if result.data:
        print("  example:", result.data[0])


with NostradamusClient(api_key=READ_KEY, base_url=BASE_URL) as read_client:
    # All records, grouped per key
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
    )
    summary(result, "📥 All records")
    counts = Counter(row["key"] for row in result.data)
    print("   per key:", dict(sorted(counts.items())))

    # 🚚 TRUCK_001 readings
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        filters=[
            {"property_name": "key", "operator": "eq", "property_value": "TRUCK_001"},
        ],
    )
    summary(result, "🚚 TRUCK_001 readings")

    # 💨 Speeding (> 90 km/h)
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        filters=[
            {"property_name": "speed_kmh", "operator": "gt", "property_value": 90},
        ],
    )
    summary(result, "💨 Speeding (> 90 km/h)")

    # 🔥 Overheating (engine temp > 100 °C)
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        filters=[
            {"property_name": "engine_temperature_c", "operator": "gt", "property_value": 100},
        ],
    )
    summary(result, "🔥 Overheating (engine temp > 100 °C)")

    # ⛽ Low fuel (< 20 %)
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        filters=[
            {"property_name": "fuel_level_pct", "operator": "lt", "property_value": 20},
        ],
    )
    summary(result, "⛽ Low fuel (< 20 %)")

    # Top 5 fuel consumption
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        attributes=["key", "fuel_consumption_lph"],
        order_by='{"field": "fuel_consumption_lph", "order": "desc"}',
        limit=5,
    )
    print("🏆 Top 5 fuel consumption")
    for row in result.data:
        print("   >", row)

## Step 4: Statistics

`data.statistics()` takes `stat`, `attribute` and optional `group_by` /
`interval` / `limit`. Use the `StatOperation` enum instead of raw strings.

In [ ]:
with NostradamusClient(api_key=READ_KEY, base_url=BASE_URL) as read_client:
    for label, attribute, stat in [
        ("Average speed (km/h)", "speed_kmh", StatOperation.AVG),
        ("Maximum engine temperature (°C)", "engine_temperature_c", StatOperation.MAX),
        ("Minimum fuel level (%)", "fuel_level_pct", StatOperation.MIN),
        ("Distinct vehicles", "key", StatOperation.DISTINCT),
    ]:
        result = read_client.data.statistics(
            project_id=PROJECT_ID,
            collection_id=collection_id,
            stat=stat,
            attribute=attribute,
        )
        print(f"📈 {label}: {result.get('value', result)}")

## Step 5: Delete a 12-hour window for one vehicle

`data.delete()` accepts `key`, `timestamp_from` and `timestamp_to` (at least one
required). Here we remove the first half of the 24-hour window for
`TRUCK_001` and compare the record count before and after.

In [ ]:
timestamp_from = (datetime.now(timezone.utc) - timedelta(hours=24)).strftime(
    "%Y-%m-%dT%H:%M:%SZ"
)
timestamp_to = (datetime.now(timezone.utc) - timedelta(hours=12)).strftime(
    "%Y-%m-%dT%H:%M:%SZ"
)

DELETE_KEY = "TRUCK_001"


with NostradamusClient(api_key=MASTER_KEY, base_url=BASE_URL) as master_client:

    def count_for(key):
        result = master_client.data.get(
            project_id=PROJECT_ID,
            collection_id=collection_id,
            filters=[{"property_name": "key", "operator": "eq", "property_value": key}],
        )
        return result.total_count

    before = count_for(DELETE_KEY)
    result = master_client.data.delete(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        key=DELETE_KEY,
        timestamp_from=timestamp_from,
        timestamp_to=timestamp_to,
    )
    after = count_for(DELETE_KEY)

print(f"🗑️  {result.message} (criteria={result.criteria})")
print(f"   {DELETE_KEY}: {before} → {after} record(s)")

## Step 6: Clean up

Remove the demo collection so the notebook can be re-run from scratch.

In [ ]:
with NostradamusClient(api_key=MASTER_KEY, base_url=BASE_URL) as master_client:
    master_client.collections.delete(
        project_id=PROJECT_ID,
        collection_id=collection_id,
    )
print(f"🧹 Collection {collection_id} deleted")
print("✨ Demo complete!")